# Figure 6D — Tumor cell cytokine and chemokine expression in MHC II high vs low LUAD

**Goal:** Characterize cytokine and chemokine expression in MHC II high vs
low tumor cells to identify tumor-intrinsic signals that may drive differences
in immune infiltration. Key question: do MHC II high tumor cells actively
recruit APCs and T cells through chemokine production?

**Approach:** Two complementary analyses:
1. Targeted analysis — hypothesis-driven gene sets focused on immune
   recruitment chemokines, with sample-level aggregation and BH FDR correction
2. Broad heatmap — all cytokines/chemokines tested, row-normalized expression
   heatmap showing the full landscape of significant findings

**Key finding:** CCL2 is significantly enriched in MHC II high tumor cells
(*** FDR-corrected), suggesting tumor-intrinsic chemokine production drives
macrophage recruitment in MHC II high tumors.

**Input:** `outputs/processed/luad_mhc2_classified.h5ad`
**Output:** Figure 6D saved to `outputs/figures/figure6/`

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from pathlib import Path
import yaml

from ceiba.plot_utils import plot_dual_metric_panel

sns.set(font_scale=1.8)
sns.set_style('ticks')
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

palette    = {'MHC class II High': '#FF8811FF', 'MHC class II Low': '#462255FF'}
group_order = ['MHC class II High', 'MHC class II Low']

In [ ]:
with open('/home/gh8sj/projects/mhc2-luad-paper/data/paths_config.yml') as f:
    cfg = yaml.safe_load(f)

repo_root = Path(cfg['repo_root'])
fig_out   = repo_root / cfg['outputs']['figures'] / 'figure6'
table_out = repo_root / cfg['outputs']['tables']  / 'figure6'

fig_out.mkdir(parents=True, exist_ok=True)
table_out.mkdir(parents=True, exist_ok=True)

## Load Data

In [ ]:
tumordata = sc.read_h5ad(repo_root / 'outputs/processed/luad_mhc2_classified.h5ad')

tumordata = tumordata[
    tumordata.obs['MHC2_clustering'].isin(['MHC class II High', 'MHC class II Low'])
].copy()

# subset to tumor cells only
tumor_cells = tumordata[tumordata.obs['cell_type_major'] == 'Tumor cells'].copy()
print(f"Tumor cells: {tumor_cells.n_obs:,}")
print(tumor_cells.obs['MHC2_clustering'].value_counts())

## Cytokine and chemokine gene sets

Full cytokine/chemokine panel organized by functional category.
Analysis uses sample-level aggregation (mean per sample) to avoid
pseudoreplication, with BH FDR correction applied within each
functional category separately.

The full panel is tested first to identify significant genes,
then a targeted primary figure is built around the significant findings.

In [ ]:
cytokine_groups = {
    'pro_inflammatory': [
        'IFNG', 'TNF', 'IL1A', 'IL1B', 'IL2', 'IL6', 'IL12A', 'IL12B', 'IL18'
    ],
    'anti_inflammatory': [
        'IL10', 'TGFB1', 'TGFB2', 'TGFB3', 'IL4', 'IL13', 'IL19', 'IL24'
    ],
    'th17_associated': [
        'IL17A', 'IL17F', 'IL21', 'IL22', 'IL23A'
    ],
    'growth_factors': [
        'CSF1', 'CSF2', 'CSF3', 'EGF', 'FGF2', 'HGF', 'VEGFA', 'PDGFA', 'PDGFB'
    ],
    'cxc_chemokines': [
        'CXCL1', 'CXCL2', 'CXCL3', 'CXCL5', 'CXCL6', 'CXCL8',
        'CXCL9', 'CXCL10', 'CXCL11', 'CXCL12', 'CXCL13', 'CXCL14'
    ],
    'cc_chemokines': [
        'CCL2', 'CCL3', 'CCL4', 'CCL5', 'CCL7', 'CCL8', 'CCL13',
        'CCL17', 'CCL19', 'CCL20', 'CCL21', 'CCL22', 'CCL25', 'CCL27', 'CCL28'
    ],
    'cx3c_xc': ['CX3CL1', 'XCL1', 'XCL2'],
    'ifn_signaling': [
        'IFNGR1', 'IFNGR2',
        'IL12RB1', 'IL12RB2',
    ],
    'tgf_signaling': [
        'TGFB1', 'TGFB2', 'TGFB3',
        'TGFBR1', 'TGFBR2',
    ],
    'receptors_other': [
        'IL13RA2', 'IL20RB',
        'CSF1R', 'CSF2RA', 'CSF2RB', 'CSF3R',
        'EGFR', 'FGFR1', 'FGFR2',
        'MET', 'KDR', 'FLT1',
        'PDGFRA', 'PDGFRB',
        'CXCR1', 'CXCR2', 'CXCR3', 'CXCR4', 'CXCR5', 'CXCR6',
        'CCR1', 'CCR2', 'CCR3', 'CCR4', 'CCR5', 'CCR6', 'CCR7',
        'CX3CR1', 'XCR1',
    ],
}

# note: TGFB1/2/3 appear in both anti_inflammatory and tgf_signaling
# deduplicate across groups to avoid double-counting in stats
seen = set()
cytokine_genes_dict = {}
missing = []
for grp, genes in cytokine_groups.items():
    for g in genes:
        if g in seen:
            continue
        seen.add(g)
        match = tumor_cells.var[tumor_cells.var['feature_name'] == g]
        if len(match) > 0:
            cytokine_genes_dict[g] = list(match.index)
        else:
            missing.append(g)

print(f"Mapped: {len(cytokine_genes_dict)} genes")
print(f"Missing: {missing}")

## Statistical testing

Per-sample mean expression is computed for each gene in each MHC II group.
Mann-Whitney U test is applied at the sample level (n=56 high, n=66 low).
FDR correction applied within each functional category separately to
reduce the correction burden while maintaining biological interpretability.

In [ ]:
all_stats = []
cytokine_pct  = {}
cytokine_mean = {}

for gene, gene_ens_list in cytokine_genes_dict.items():
    gene_ens = gene_ens_list[0]
    x = tumor_cells.to_df()[gene_ens]
    expr = x.to_frame(name='expr')
    expr['sample']  = tumor_cells.obs['sample'].values
    expr['cluster'] = tumor_cells.obs['MHC2_clustering'].values

    # % expressing
    pct_df = (
        expr.assign(detected=(expr['expr'] > 0).astype(int))
        .groupby(['sample', 'cluster'], observed=True)['detected']
        .mean().mul(100).reset_index()
        .rename(columns={'detected': 'expr'})
    )
    cytokine_pct[gene] = pct_df

    # mean among positive cells
    mean_df = (
        expr[expr['expr'] > 0]
        .groupby(['sample', 'cluster'], observed=True)['expr']
        .mean().reset_index()
    )
    cytokine_mean[gene] = mean_df

    g1_pct = pct_df.loc[pct_df['cluster'] == group_order[0], 'expr']
    g2_pct = pct_df.loc[pct_df['cluster'] == group_order[1], 'expr']
    p_pct  = mannwhitneyu(g1_pct, g2_pct, alternative='two-sided')[1] if len(g1_pct) > 0 and len(g2_pct) > 0 else np.nan

    g1_mean = mean_df.loc[mean_df['cluster'] == group_order[0], 'expr']
    g2_mean = mean_df.loc[mean_df['cluster'] == group_order[1], 'expr']
    p_mean  = mannwhitneyu(g1_mean, g2_mean, alternative='two-sided')[1] if len(g1_mean) > 0 and len(g2_mean) > 0 else np.nan

    grp_label = next((k for k, v in cytokine_groups.items() if gene in v), 'other')
    all_stats.append({
        'gene': gene, 'group': grp_label,
        'p_pct': p_pct, 'p_mean': p_mean,
        'mean_high_pct': g1_pct.mean(), 'mean_low_pct': g2_pct.mean(),
        'mean_high_expr': g1_mean.mean(), 'mean_low_expr': g2_mean.mean(),
    })

stats_df = pd.DataFrame(all_stats)

# FDR within % and avg expression separately
fdr_results = []
for grp, sub in stats_df.groupby('group', observed=True):
    sub = sub.copy()
    _, fdr_pct,  _, _ = multipletests(sub['p_pct'].fillna(1),  method='fdr_bh')
    _, fdr_mean, _, _ = multipletests(sub['p_mean'].fillna(1), method='fdr_bh')
    sub['FDR_pct']  = fdr_pct
    sub['FDR_mean'] = fdr_mean
    fdr_results.append(sub)

stats_df = pd.concat(fdr_results)
stats_df['sig_pct']  = stats_df['FDR_pct'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
stats_df['sig_mean'] = stats_df['FDR_mean'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')
stats_df['direction'] = stats_df.apply(
    lambda r: 'MHC II High' if r['mean_high_pct'] > r['mean_low_pct'] else 'MHC II Low', axis=1)

stats_df = stats_df.sort_values('FDR_pct')
stats_df.to_csv(table_out / 'fig6d_tumor_cytokine_stats.csv', index=False)

sig = stats_df[(stats_df['sig_pct'] != '') | (stats_df['sig_mean'] != '')]
print(f"Significant findings: {len(sig)}")
print(sig[['gene', 'group', 'sig_pct', 'sig_mean', 'direction']].to_string(index=False))

## Supplementary — Full cytokine and chemokine panel

Complete results for all tested cytokine and chemokine genes in tumor
cells, shown as two independent lollipop plots. Each point represents
one gene; color indicates which MHC II group shows higher expression;
filled larger points are FDR significant (FDR < 0.05); the dashed
vertical line marks the FDR = 0.05 threshold.

Left figure is ordered by significance of percent expressing metric.
Right figure is independently ordered by significance of mean expression
among positive cells — capturing genes where the level of expression
differs even when detection frequency is similar between groups.

FDR correction was applied within each functional gene category
separately (pro-inflammatory, anti-inflammatory, CXC chemokines, etc.)
to reduce the correction burden while maintaining biological
interpretability. Genes are colored by direction of effect regardless
of significance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, len(supp_df) * 0.3 + 2), sharey=False)

for ax, (metric, col, sig_col) in zip(axes, [
    ('% cells expressing',          '-log10_FDR_pct',  'sig_pct'),
    ('Mean expression (pos cells)', '-log10_FDR_mean', 'sig_mean'),
]):
    plot_df = supp_df.sort_values(col, ascending=True).copy()

    for _, row in plot_df.iterrows():
        color  = palette['MHC class II High'] if row['direction'] == 'MHC II High' else palette['MHC class II Low']
        sig    = row[sig_col] != ''
        marker = 'o' if sig else 'x'
        size   = 80 if sig else 50

        ax.plot([0, row[col]], [row['gene'], row['gene']],
                color=color, lw=1, alpha=0.6)
        ax.scatter(row[col], row['gene'],
                   color=color, s=size, marker=marker,
                   zorder=3, linewidths=1)

    ax.axvline(-np.log10(0.05), color='gray', linestyle='--', lw=1)
    ax.set_xlabel('-log10(FDR-adjusted p-value)', fontsize=11)
    ax.set_title(metric, fontsize=12)
    ax.spines[['top', 'right']].set_visible(False)

legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=palette['MHC class II High'],
           markersize=10, label='Higher in MHC II High'),
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=palette['MHC class II Low'],
           markersize=10, label='Higher in MHC II Low'),
    Line2D([0], [0], marker='o', color='gray',
           markerfacecolor='gray', markersize=10, label='FDR < 0.05'),
    Line2D([0], [0], marker='x', color='gray',
            markersize=8, label='ns', linewidth=0),
]
fig.legend(handles=legend_elements, loc='upper center', ncol=4,
           frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=10)

plt.tight_layout()
plt.savefig(fig_out / 'figS_tumor_cytokine_full_lollipop.pdf', bbox_inches='tight')
plt.show()

## Figure 6D — Tumor cell cytokine and chemokine expression

Primary figure showing the most biologically interpretable significant
findings from the full cytokine/chemokine panel. Two metrics are shown
for each gene: percent of tumor cells expressing the gene per sample
(top row) and mean expression level among expressing cells (bottom row).
FDR correction applied within each functional category separately.

Key findings:
- **CCL2** (***) — higher in MHC II high. Tumor-intrinsic macrophage
  recruitment signal. Creates a potential self-reinforcing loop: MHC II
  high tumor cells produce CCL2 → recruit macrophages → macrophages
  upregulate MHC II and co-stimulatory molecules → drive T cell activation.
- **IFNGR1** (***) — higher in MHC II high. MHC II high tumor cells are
  more responsive to IFN-γ signaling, which drives CIITA → MHC II
  upregulation via the canonical pathway. Provides a mechanistic link
  between immune activity and tumor MHC II expression.
- **IL18** (**) — higher in MHC II high. Inflammasome activation marker,
  consistent with a more immunologically active tumor microenvironment.
- **TGFB2** (**) — higher in MHC II high. May reflect a regulatory
  feedback response to an active immune environment rather than
  immunosuppression per se.
- **CXCL5** (*) — higher in MHC II low. Neutrophil-recruiting chemokine
  via CXCR2. Supports the divergent immune response model — MHC II low
  tumors drive innate inflammatory recruitment rather than adaptive
  recognition-based responses.
- **CSF3R** (**) — higher in MHC II low at expression level. G-CSF
  receptor on tumor cells; higher expression suggests MHC II low tumor
  cells are more responsive to G-CSF signaling, which promotes
  neutrophil differentiation and tumor-promoting inflammation.
- **CCL20** (*) — higher in MHC II low at expression level. Recruits
  immature DCs and Th17 cells; may reflect a failed attempt to recruit
  DCs that lack the activation signals to mature properly.

In [ ]:
primary_genes = ['CCL2', 'IFNGR1', 'IL18', 'TGFB2', 'CXCL5', 'CSF3R', 'CCL20']
primary_genes_dict = {g: cytokine_genes_dict[g] for g in primary_genes if g in cytokine_genes_dict}

# build combined pct + mean data for primary genes
primary_pct  = {g: cytokine_pct[g]  for g in primary_genes}
primary_mean = {g: cytokine_mean[g] for g in primary_genes}

nrows = 2  # row 1: % expressing, row 2: mean expression
ncols = len(primary_genes)

fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 4 * nrows), sharey=False)
axes = np.atleast_2d(axes)

for c, gene in enumerate(primary_genes):
    sig_p = stats_df.loc[stats_df['gene'] == gene, 'sig_pct'].values[0]
    sig_m = stats_df.loc[stats_df['gene'] == gene, 'sig_mean'].values[0]
    direction = stats_df.loc[stats_df['gene'] == gene, 'direction'].values[0]

    for r, (metric, data, sig) in enumerate([
        ('% cells expressing', primary_pct[gene],  sig_p),
        ('mean expr\n(pos cells)', primary_mean[gene], sig_m),
    ]):
        ax = axes[r, c]
        sns.boxplot(
            data=data, x='cluster', y='expr', hue='cluster',
            order=group_order, palette=palette,
            ax=ax, fill=False, showfliers=False, legend=False,
        )
        sns.stripplot(
            data=data, x='cluster', y='expr', hue='cluster',
            order=group_order, palette=palette,
            ax=ax, edgecolor='k', linewidth=1,
            size=4, alpha=0.7, legend=False,
        )
        if r == 0:
            ax.set_title(gene, fontsize=18, pad=12)
        if c == 0:
            ax.set_ylabel(metric, fontsize=13)
        else:
            ax.set_ylabel('')
        ax.set_xlabel('')
        ax.set_xticklabels([])
        ax.spines[['top', 'right']].set_visible(False)
        if sig:
            ax.text(0.5, 0.80, sig, ha='center', va='bottom',
                    transform=ax.transAxes, fontsize=36)

handles = [
    plt.Line2D([0], [0], color=palette[g], marker='o', linestyle='',
               markersize=8, markeredgecolor='k', markeredgewidth=1, label=g)
    for g in group_order
]
fig.legend(handles=handles, loc='upper center', ncol=2,
           frameon=False, bbox_to_anchor=(0.5, 1.02), fontsize=14)
fig.suptitle('Tumor cell cytokine and chemokine expression by MHC II classification',
             fontsize=15, y=1.05)
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(fig_out / 'fig6d_tumor_cytokines_primary.pdf', bbox_inches='tight')
plt.show()